In [1]:
# ================================================================
# MODEL 2 — WHAT CHANGED?
# 4-CLASS SEMANTIC CHANGE CLASSIFICATION
#
# Classes:
#   0 = Building / Construction
#   1 = Vegetation
#   2 = Road / Infrastructure
#   3 = Other Change
#
# Input:
#   LEVIR-CC image pair (A, B)
#
# Supervision:
#   Weak labels derived from all available LEVIR-CC captions
#
# Architecture:
#   Siamese ResNet18
#   A + B + |A-B| + A*B
#   -> MLP -> 4 classes
#
# Primary metric:
#   Macro F1
# ================================================================

import os
import ast
import json
import re
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt


# ================================================================
# 1. REPRODUCIBILITY
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ================================================================
# 2. PATHS
# ================================================================

BASE_DIR = Path(
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
)

CSV_PATH = BASE_DIR / "final_dataset.csv"

OUTPUT_DIR = (
    BASE_DIR /
    "CHANGE_CLASSIFICATION_MODEL"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH = (
    OUTPUT_DIR /
    "model2_4class_best_siamese_resnet18.pth"
)

LABELED_CSV_PATH = (
    OUTPUT_DIR /
    "model2_4class_labeled_dataset.csv"
)

REPORT_PATH = (
    OUTPUT_DIR /
    "model2_4class_classification_report.txt"
)

CM_PATH = (
    OUTPUT_DIR /
    "model2_4class_confusion_matrix.png"
)

HISTORY_PATH = (
    OUTPUT_DIR /
    "model2_4class_training_history.csv"
)

CONFIG_PATH = (
    OUTPUT_DIR /
    "model2_4class_config.json"
)


# ================================================================
# 3. CONFIGURATION
# ================================================================

IMAGE_SIZE = 224

BATCH_SIZE = 32

EPOCHS = 30

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

PATIENCE = 8

NUM_CLASSES = 4

NUM_WORKERS = 0

USE_AMP = True


# ================================================================
# 4. DEVICE
# ================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("MODEL 2 — WHAT CHANGED?")
print("4-CLASS SEMANTIC CHANGE CLASSIFICATION")
print("=" * 72)

print(f"Device: {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print()


# ================================================================
# 5. CLASS DEFINITIONS
# ================================================================

CLASS_NAMES = [
    "Building / Construction",
    "Vegetation",
    "Road / Infrastructure",
    "Other Change"
]

CLASS_TO_ID = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}


# ================================================================
# 6. KEYWORDS
# ================================================================

BUILDING_KEYWORDS = [
    "building",
    "buildings",
    "house",
    "houses",
    "home",
    "homes",
    "residential",
    "commercial",
    "construction",
    "constructed",
    "structure",
    "structures",
    "apartment",
    "apartments",
    "warehouse",
    "warehouses",
    "industrial",
    "factory",
    "factories",
    "roof",
    "roofs",
    "urban",
    "development",
    "developed",
    "housing",
    "complex",
    "school",
    "hospital"
]

VEGETATION_KEYWORDS = [
    "vegetation",
    "vegetated",
    "tree",
    "trees",
    "forest",
    "forests",
    "woodland",
    "woods",
    "grass",
    "grasses",
    "greenery",
    "plant",
    "plants",
    "crop",
    "crops",
    "farmland",
    "field",
    "fields",
    "agriculture",
    "agricultural",
    "orchard",
    "orchards",
    "garden",
    "gardens",
    "bush",
    "bushes",
    "shrub",
    "shrubs",
    "park",
    "parks"
]

ROAD_KEYWORDS = [
    "road",
    "roads",
    "highway",
    "highways",
    "street",
    "streets",
    "path",
    "paths",
    "driveway",
    "driveways",
    "railway",
    "railroad",
    "rail",
    "bridge",
    "bridges",
    "parking",
    "parking lot",
    "parkinglot",
    "runway",
    "runways",
    "airport",
    "intersection",
    "junction",
    "lane",
    "lanes",
    "pavement",
    "paved"
]


# ================================================================
# 7. TEXT NORMALIZATION
# ================================================================

def normalize_text(text):

    if text is None:
        return ""

    text = str(text).lower()

    text = text.replace("_", " ")
    text = text.replace("-", " ")

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ================================================================
# 8. CAPTION PARSER
# ================================================================

def parse_captions(value):

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, list):

        return [
            str(x).strip()
            for x in value
            if str(x).strip()
        ]

    value = str(value).strip()

    if not value:
        return []

    # Python-list representation
    try:

        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):

            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

        if isinstance(parsed, str):

            return [parsed]

    except Exception:
        pass

    # JSON representation
    try:

        parsed = json.loads(value)

        if isinstance(parsed, list):

            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except Exception:
        pass

    return [value]


# ================================================================
# 9. KEYWORD SCORE
# ================================================================

def keyword_score(text, keywords):

    text = normalize_text(text)

    score = 0

    for keyword in keywords:

        keyword = normalize_text(keyword)

        pattern = (
            r"\b"
            + re.escape(keyword)
            + r"\b"
        )

        score += len(
            re.findall(
                pattern,
                text
            )
        )

    return score


# ================================================================
# 10. SEMANTIC LABEL GENERATION
# ================================================================

def generate_semantic_label(caption_value):

    captions = parse_captions(
        caption_value
    )

    building_score = 0
    vegetation_score = 0
    road_score = 0

    for caption in captions:

        building_score += keyword_score(
            caption,
            BUILDING_KEYWORDS
        )

        vegetation_score += keyword_score(
            caption,
            VEGETATION_KEYWORDS
        )

        road_score += keyword_score(
            caption,
            ROAD_KEYWORDS
        )

    scores = {
        "building": building_score,
        "vegetation": vegetation_score,
        "road": road_score
    }

    # No semantic keyword detected.
    # Assign to Other Change.
    if max(scores.values()) == 0:

        return (
            3,
            "Other Change",
            scores
        )

    # Strongest semantic evidence
    max_score = max(
        scores.values()
    )

    winners = [
        key
        for key, value in scores.items()
        if value == max_score
    ]

    # Deterministic tie-breaking
    if "building" in winners:

        return (
            0,
            "Building / Construction",
            scores
        )

    if "vegetation" in winners:

        return (
            1,
            "Vegetation",
            scores
        )

    return (
        2,
        "Road / Infrastructure",
        scores
    )


# ================================================================
# 11. LOAD DATASET
# ================================================================

print("=" * 72)
print("LOADING DATASET")
print("=" * 72)

df = pd.read_csv(
    CSV_PATH
)

print(
    f"Rows loaded: {len(df):,}"
)

required_columns = [
    "split",
    "cc_A",
    "cc_B",
    "captions"
]

for column in required_columns:

    if column not in df.columns:

        raise RuntimeError(
            f"Required column missing: {column}"
        )


# ================================================================
# 12. REMOVE DUPLICATE IMAGE PAIRS
# ================================================================

df["pair_id"] = (
    df["cc_A"].astype(str)
    + "||"
    + df["cc_B"].astype(str)
)

before = len(df)

df = df.drop_duplicates(
    subset="pair_id"
).copy()

print(
    f"Unique CC image pairs: {len(df):,}"
)

print(
    f"Duplicates removed: {before - len(df):,}"
)


# ================================================================
# 13. GENERATE LABELS
# ================================================================

print()
print("=" * 72)
print("GENERATING WEAK SEMANTIC LABELS")
print("=" * 72)

labels = []
class_names = []

building_scores = []
vegetation_scores = []
road_scores = []

for captions in df["captions"]:

    label_id, label_name, scores = (
        generate_semantic_label(
            captions
        )
    )

    labels.append(label_id)

    class_names.append(
        label_name
    )

    building_scores.append(
        scores["building"]
    )

    vegetation_scores.append(
        scores["vegetation"]
    )

    road_scores.append(
        scores["road"]
    )


df["semantic_label"] = labels

df["semantic_class"] = class_names

df["building_score"] = building_scores

df["vegetation_score"] = vegetation_scores

df["road_score"] = road_scores


# ================================================================
# 14. LABEL DISTRIBUTION
# ================================================================

print()
print("=" * 72)
print("OVERALL LABEL DISTRIBUTION")
print("=" * 72)

overall_counts = (
    df["semantic_class"]
    .value_counts()
)

for class_name in CLASS_NAMES:

    count = int(
        overall_counts.get(
            class_name,
            0
        )
    )

    percentage = (
        100 * count / len(df)
    )

    print(
        f"{class_name:<30}"
        f"{count:>6} "
        f"({percentage:6.2f}%)"
    )


# ================================================================
# 15. SPLIT DISTRIBUTION
# ================================================================

print()
print("=" * 72)
print("CLASS DISTRIBUTION BY SPLIT")
print("=" * 72)

distribution = pd.crosstab(
    df["split"],
    df["semantic_class"]
)

distribution = distribution.reindex(
    columns=CLASS_NAMES,
    fill_value=0
)

print(distribution)


# ================================================================
# 16. CHECK FOR MISSING CLASSES
# ================================================================

missing_classes = []

for class_name in CLASS_NAMES:

    if (
        df["semantic_class"] == class_name
    ).sum() == 0:

        missing_classes.append(
            class_name
        )

if missing_classes:

    print()
    print(
        "ERROR: The following classes have "
        "zero samples:"
    )

    for class_name in missing_classes:
        print(
            f"  - {class_name}"
        )

    raise RuntimeError(
        "Cannot train a 4-class classifier "
        "with missing classes."
    )


# ================================================================
# 17. TRAIN / VALIDATION DATA
# ================================================================

train_df = df[
    df["split"].astype(str).str.lower()
    == "train"
].copy()

val_df = df[
    df["split"].astype(str).str.lower()
    == "val"
].copy()

print()
print("=" * 72)
print("TRAIN / VALIDATION")
print("=" * 72)

print(
    f"Train: {len(train_df):,}"
)

print(
    f"Validation: {len(val_df):,}"
)


# ================================================================
# 18. CHECK THAT EVERY CLASS EXISTS IN TRAIN AND VAL
# ================================================================

for class_id, class_name in enumerate(
    CLASS_NAMES
):

    train_count = (
        train_df["semantic_label"]
        == class_id
    ).sum()

    val_count = (
        val_df["semantic_label"]
        == class_id
    ).sum()

    print(
        f"{class_id} | "
        f"{class_name:<30} | "
        f"Train: {train_count:>5} | "
        f"Val: {val_count:>5}"
    )

    if train_count == 0:

        raise RuntimeError(
            f"{class_name} has no training samples."
        )

    if val_count == 0:

        raise RuntimeError(
            f"{class_name} has no validation samples."
        )


# ================================================================
# 19. SAVE LABELED DATASET
# ================================================================

df.to_csv(
    LABELED_CSV_PATH,
    index=False
)

print()
print(
    f"Labeled dataset saved:\n"
    f"{LABELED_CSV_PATH}"
)


# ================================================================
# 20. PATH RESOLUTION
# ================================================================

def resolve_path(value):

    if pd.isna(value):
        return None

    path = Path(
        str(value)
    )

    # Absolute path
    if path.is_absolute():

        if path.exists():
            return path

    # Relative to dataset root
    candidate = (
        BASE_DIR / path
    )

    if candidate.exists():
        return candidate

    # Direct fallback
    if path.exists():
        return path

    return None


# ================================================================
# 21. VERIFY IMAGE PATHS
# ================================================================

print()
print("=" * 72)
print("VERIFYING IMAGE PATHS")
print("=" * 72)

sample = df.sample(
    min(100, len(df)),
    random_state=SEED
)

missing_A = 0
missing_B = 0

for _, row in sample.iterrows():

    if resolve_path(
        row["cc_A"]
    ) is None:

        missing_A += 1

    if resolve_path(
        row["cc_B"]
    ) is None:

        missing_B += 1

print(
    f"Missing A: {missing_A}"
)

print(
    f"Missing B: {missing_B}"
)

if missing_A > 0 or missing_B > 0:

    raise RuntimeError(
        "Image path verification failed."
    )

print(
    "Image path verification PASSED."
)


# ================================================================
# 22. DATASET
# ================================================================

class ChangeClassificationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        path_A = resolve_path(
            row["cc_A"]
        )

        path_B = resolve_path(
            row["cc_B"]
        )

        if path_A is None:

            raise FileNotFoundError(
                f"A image not found:\n"
                f"{row['cc_A']}"
            )

        if path_B is None:

            raise FileNotFoundError(
                f"B image not found:\n"
                f"{row['cc_B']}"
            )

        image_A = Image.open(
            path_A
        ).convert("RGB")

        image_B = Image.open(
            path_B
        ).convert("RGB")

        image_A = self.transform(
            image_A
        )

        image_B = self.transform(
            image_B
        )

        label = int(
            row["semantic_label"]
        )

        return (
            image_A,
            image_B,
            label
        )


# ================================================================
# 23. TRANSFORMS
# ================================================================

weights = ResNet18_Weights.DEFAULT

imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std


train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.5
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])


val_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])


# ================================================================
# 24. DATALOADERS
# ================================================================

train_dataset = ChangeClassificationDataset(
    train_df,
    train_transform
)

val_dataset = ChangeClassificationDataset(
    val_df,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    )
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    )
)

print()
print(
    f"Train batches: {len(train_loader)}"
)

print(
    f"Val batches: {len(val_loader)}"
)


# ================================================================
# 25. CLASS WEIGHTS
# ================================================================

train_labels = (
    train_df["semantic_label"]
    .astype(int)
    .values
)

class_counts = np.bincount(
    train_labels,
    minlength=NUM_CLASSES
)

total = class_counts.sum()

class_weights = (
    total
    /
    (
        NUM_CLASSES
        *
        np.maximum(
            class_counts,
            1
        )
    )
)

# Normalize mean weight to 1
class_weights = (
    class_weights
    /
    class_weights.mean()
)

print()
print("=" * 72)
print("CLASS WEIGHTS")
print("=" * 72)

for i, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"{i} | "
        f"{class_name:<30} | "
        f"Count: {class_counts[i]:>5} | "
        f"Weight: {class_weights[i]:.4f}"
    )

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


# ================================================================
# 26. MODEL
# ================================================================

class SiameseResNet18(
    nn.Module
):

    def __init__(
        self,
        num_classes=4
    ):

        super().__init__()

        backbone = models.resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        self.encoder = nn.Sequential(
            *list(
                backbone.children()
            )[:-1]
        )

        feature_dim = 512

        combined_dim = (
            feature_dim * 4
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                combined_dim,
                512
            ),

            nn.BatchNorm1d(
                512
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                512,
                128
            ),

            nn.BatchNorm1d(
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                0.20
            ),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(
        self,
        image_A,
        image_B
    ):

        feat_A = self.encoder(
            image_A
        )

        feat_B = self.encoder(
            image_B
        )

        feat_A = torch.flatten(
            feat_A,
            start_dim=1
        )

        feat_B = torch.flatten(
            feat_B,
            start_dim=1
        )

        absolute_difference = torch.abs(
            feat_A - feat_B
        )

        elementwise_product = (
            feat_A * feat_B
        )

        combined = torch.cat(
            [
                feat_A,
                feat_B,
                absolute_difference,
                elementwise_product
            ],
            dim=1
        )

        return self.classifier(
            combined
        )


model = SiameseResNet18(
    num_classes=NUM_CLASSES
).to(DEVICE)


# ================================================================
# 27. LOSS / OPTIMIZER
# ================================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)


# ================================================================
# 28. AMP
# ================================================================

if (
    DEVICE.type == "cuda"
    and USE_AMP
):

    scaler = torch.amp.GradScaler(
        "cuda"
    )

else:

    scaler = None


# ================================================================
# 29. TRAIN ONE EPOCH
# ================================================================

def train_one_epoch():

    model.train()

    total_loss = 0.0

    predictions_all = []
    labels_all = []

    for image_A, image_B, labels in train_loader:

        image_A = image_A.to(
            DEVICE,
            non_blocking=True
        )

        image_B = image_B.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        if scaler is not None:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    image_A,
                    image_B
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            logits = model(
                image_A,
                image_B
            )

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            optimizer.step()

        total_loss += (
            loss.item()
            * labels.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        predictions_all.extend(
            predictions.detach()
            .cpu()
            .numpy()
        )

        labels_all.extend(
            labels.detach()
            .cpu()
            .numpy()
        )

    epoch_loss = (
        total_loss
        /
        len(train_dataset)
    )

    epoch_f1 = f1_score(
        labels_all,
        predictions_all,
        average="macro",
        zero_division=0
    )

    epoch_accuracy = accuracy_score(
        labels_all,
        predictions_all
    )

    return (
        epoch_loss,
        epoch_f1,
        epoch_accuracy
    )


# ================================================================
# 30. VALIDATION
# ================================================================

@torch.no_grad()
def validate():

    model.eval()

    total_loss = 0.0

    predictions_all = []
    labels_all = []

    for image_A, image_B, labels in val_loader:

        image_A = image_A.to(
            DEVICE,
            non_blocking=True
        )

        image_B = image_B.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if (
            DEVICE.type == "cuda"
            and USE_AMP
        ):

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    image_A,
                    image_B
                )

                loss = criterion(
                    logits,
                    labels
                )

        else:

            logits = model(
                image_A,
                image_B
            )

            loss = criterion(
                logits,
                labels
            )

        total_loss += (
            loss.item()
            * labels.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        predictions_all.extend(
            predictions.cpu().numpy()
        )

        labels_all.extend(
            labels.cpu().numpy()
        )

    val_loss = (
        total_loss
        /
        len(val_dataset)
    )

    accuracy = accuracy_score(
        labels_all,
        predictions_all
    )

    precision = precision_score(
        labels_all,
        predictions_all,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        labels_all,
        predictions_all,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        labels_all,
        predictions_all,
        average="macro",
        zero_division=0
    )

    return (
        val_loss,
        accuracy,
        precision,
        recall,
        macro_f1,
        np.array(labels_all),
        np.array(predictions_all)
    )


# ================================================================
# 31. TRAINING
# ================================================================

print()
print("=" * 72)
print("STARTING MODEL 2 TRAINING")
print("=" * 72)

best_macro_f1 = -1.0

best_epoch = 0

epochs_without_improvement = 0

history = []

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    (
        train_loss,
        train_f1,
        train_accuracy
    ) = train_one_epoch()

    (
        val_loss,
        val_accuracy,
        val_precision,
        val_recall,
        val_f1,
        val_labels,
        val_predictions
    ) = validate()

    scheduler.step(
        val_f1
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    history.append({

        "epoch": epoch,

        "train_loss": train_loss,

        "train_macro_f1": train_f1,

        "train_accuracy": train_accuracy,

        "val_loss": val_loss,

        "val_macro_f1": val_f1,

        "val_accuracy": val_accuracy,

        "val_macro_precision":
            val_precision,

        "val_macro_recall":
            val_recall,

        "learning_rate":
            current_lr,

        "epoch_time_sec":
            epoch_time
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train F1 {train_f1:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Macro-F1 {val_f1:.4f} | "
        f"Val Acc {val_accuracy:.4f} | "
        f"LR {current_lr:.2e} | "
        f"{epoch_time:.1f}s"
    )

    # ------------------------------------------------------------
    # BEST CHECKPOINT
    # ------------------------------------------------------------

    if val_f1 > best_macro_f1:

        best_macro_f1 = val_f1

        best_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "best_val_macro_f1":
                    best_macro_f1,

                "class_names":
                    CLASS_NAMES,

                "class_weights":
                    class_weights.tolist(),

                "config": {
                    "image_size":
                        IMAGE_SIZE,

                    "batch_size":
                        BATCH_SIZE,

                    "epochs":
                        EPOCHS,

                    "learning_rate":
                        LEARNING_RATE,

                    "weight_decay":
                        WEIGHT_DECAY,

                    "num_classes":
                        NUM_CLASSES,

                    "seed":
                        SEED
                }
            },
            CHECKPOINT_PATH
        )

        print(
            f"    ★ BEST MODEL SAVED "
            f"| Macro-F1 = {best_macro_f1:.4f}"
        )

    else:

        epochs_without_improvement += 1

    # ------------------------------------------------------------
    # EARLY STOPPING
    # ------------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print()
        print(
            f"Early stopping after "
            f"{PATIENCE} epochs without "
            f"Macro-F1 improvement."
        )

        break


# ================================================================
# 32. TRAINING SUMMARY
# ================================================================

training_time = (
    time.time()
    - training_start
)

print()
print("=" * 72)
print("TRAINING COMPLETE")
print("=" * 72)

print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best validation Macro-F1: "
    f"{best_macro_f1:.6f}"
)

print(
    f"Training time: "
    f"{training_time / 60:.2f} minutes"
)


# ================================================================
# 33. SAVE HISTORY
# ================================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)


# ================================================================
# 34. LOAD BEST CHECKPOINT
# ================================================================

print()
print("=" * 72)
print("LOADING BEST CHECKPOINT")
print("=" * 72)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

print(
    f"Loaded epoch: "
    f"{checkpoint['epoch']}"
)

print(
    f"Stored Macro-F1: "
    f"{checkpoint['best_val_macro_f1']:.6f}"
)


# ================================================================
# 35. FINAL VALIDATION
# ================================================================

(
    final_loss,
    final_accuracy,
    final_precision,
    final_recall,
    final_macro_f1,
    final_labels,
    final_predictions
) = validate()


# ================================================================
# 36. CLASSIFICATION REPORT
# ================================================================

report = classification_report(
    final_labels,
    final_predictions,
    labels=[
        0,
        1,
        2,
        3
    ],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

print()
print("=" * 72)
print("FINAL VALIDATION RESULTS")
print("=" * 72)

print(
    f"Accuracy:        {final_accuracy:.6f}"
)

print(
    f"Macro Precision: {final_precision:.6f}"
)

print(
    f"Macro Recall:    {final_recall:.6f}"
)

print(
    f"Macro F1:        {final_macro_f1:.6f}"
)

print()
print(report)


# ================================================================
# 37. CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    final_labels,
    final_predictions,
    labels=[
        0,
        1,
        2,
        3
    ]
)

print()
print("=" * 72)
print("CONFUSION MATRIX")
print("=" * 72)

print(cm)


# ================================================================
# 38. SAVE REPORT
# ================================================================

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MODEL 2 — WHAT CHANGED?\n"
    )

    f.write(
        "4-Class Semantic Change Classification\n"
    )

    f.write(
        "=" * 72 + "\n\n"
    )

    f.write(
        f"Best Epoch: {best_epoch}\n"
    )

    f.write(
        f"Best Validation Macro-F1: "
        f"{best_macro_f1:.6f}\n"
    )

    f.write(
        f"Final Validation Accuracy: "
        f"{final_accuracy:.6f}\n"
    )

    f.write(
        f"Final Validation Macro Precision: "
        f"{final_precision:.6f}\n"
    )

    f.write(
        f"Final Validation Macro Recall: "
        f"{final_recall:.6f}\n"
    )

    f.write(
        f"Final Validation Macro-F1: "
        f"{final_macro_f1:.6f}\n\n"
    )

    f.write(
        "CLASSIFICATION REPORT\n"
    )

    f.write(
        "-" * 72 + "\n"
    )

    f.write(
        report
    )

    f.write(
        "\n\nCONFUSION MATRIX\n"
    )

    f.write(
        "-" * 72 + "\n"
    )

    f.write(
        str(cm)
    )


# ================================================================
# 39. SAVE CONFUSION MATRIX IMAGE
# ================================================================

plt.figure(
    figsize=(9, 7)
)

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.title(
    "Model 2 — Semantic Change Classification"
)

plt.colorbar()

tick_marks = np.arange(
    NUM_CLASSES
)

plt.xticks(
    tick_marks,
    CLASS_NAMES,
    rotation=35,
    ha="right"
)

plt.yticks(
    tick_marks,
    CLASS_NAMES
)

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center"
        )

plt.ylabel(
    "True Class"
)

plt.xlabel(
    "Predicted Class"
)

plt.tight_layout()

plt.savefig(
    CM_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ================================================================
# 40. SAVE CONFIG
# ================================================================

config = {

    "task":
        "Semantic Change Classification",

    "architecture":
        "Siamese ResNet18",

    "classes":
        CLASS_NAMES,

    "num_classes":
        NUM_CLASSES,

    "input":
        "LEVIR-CC paired images",

    "supervision":
        "Weak caption-derived labels",

    "feature_combination":
        "A + B + abs(A-B) + A*B",

    "primary_metric":
        "Macro F1",

    "image_size":
        IMAGE_SIZE,

    "batch_size":
        BATCH_SIZE,

    "epochs":
        EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "best_epoch":
        best_epoch,

    "best_validation_macro_f1":
        float(best_macro_f1),

    "final_validation_accuracy":
        float(final_accuracy),

    "final_validation_macro_precision":
        float(final_precision),

    "final_validation_macro_recall":
        float(final_recall),

    "final_validation_macro_f1":
        float(final_macro_f1),

    "training_time_minutes":
        float(training_time / 60)
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )


# ================================================================
# 41. FINAL OUTPUT
# ================================================================

print()
print("=" * 72)
print("MODEL 2 COMPLETE")
print("=" * 72)

print(
    f"""
Best checkpoint:
{CHECKPOINT_PATH}

Labeled dataset:
{LABELED_CSV_PATH}

Classification report:
{REPORT_PATH}

Confusion matrix:
{CM_PATH}

Training history:
{HISTORY_PATH}

Configuration:
{CONFIG_PATH}
"""
)

print("=" * 72)

MODEL 2 — WHAT CHANGED?
4-CLASS SEMANTIC CHANGE CLASSIFICATION
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU memory: 7.96 GB

LOADING DATASET
Rows loaded: 7,264
Unique CC image pairs: 7,264
Duplicates removed: 0

GENERATING WEAK SEMANTIC LABELS

OVERALL LABEL DISTRIBUTION
Building / Construction         2324 ( 31.99%)
Vegetation                       334 (  4.60%)
Road / Infrastructure           1211 ( 16.67%)
Other Change                    3395 ( 46.74%)

CLASS DISTRIBUTION BY SPLIT
semantic_class  Building / Construction  Vegetation  Road / Infrastructure  \
split                                                                        
train                              2038         289                   1076   
val                                 286          45                    135   

semantic_class  Other Change  
split                         
train                   2942  
val                      453  

TRAIN / VALIDATION
Train: 6,345
Validation: 919
0 | Building 